# 컨텍스트 로트

근거 문장의 위치와 주변 문장 수가 답에 어떤 차이를 만들까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 로컬 생성 모델 준비

고정 revision의 사전학습 모델과 tokenizer를 로컬에서 불러옵니다. `generate()`는 프롬프트 문자열과 최대 생성 토큰 수를 입력받습니다. 모델은 샘플링을 끈 같은 생성 방식으로 새 토큰 ID를 만들고, 함수는 새 부분만 문자열로 복원해 반환합니다. 모델 구조를 구현하거나 학습하는 셀이 아닙니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION)
model.eval()
print("model:", MODEL_ID)
print("parameters:", f"{model.num_parameters():,}")

model: Qwen/Qwen2.5-0.5B-Instruct
parameters: 494,032,768


In [2]:
def generate(prompt: str, *, max_new_tokens: int = 48) -> str:
    # 샘플링을 끈 결정적 생성으로 새 토큰만 복원합니다.
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 비교 조건 준비

`model.config.max_position_embeddings`가 반환한 모델의 위치 한도를 출력합니다. 정답 근거와 질문은 고정하고 무관한 문장 수를 2·20·60개로, 근거 위치를 시작·중간·끝으로 바꿉니다. 이 조합으로 아홉 개의 프롬프트를 만들어 `conditions` 목록에 추가합니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
fact = "해바라기 작전의 확인 코드는 314159입니다."
filler = "주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다."
question = "해바라기 작전의 확인 코드만 답하세요."
model_limit = model.config.max_position_embeddings
max_new_tokens = 20
print("model max_position_embeddings:", model_limit)
conditions = []
for repeats in [2, 20, 60]:
    for position in ["start", "middle", "end"]:
        items = [filler] * repeats
        index = {"start": 0, "middle": repeats // 2, "end": repeats}[position]
        items.insert(index, fact)
        prompt = "\n".join(items) + "\n\n" + question
        conditions.append((repeats, position, prompt))

model max_position_embeddings: 32768


## 아홉 조건 실행

준비한 프롬프트를 같은 결정적 생성 방식으로 두 번씩 호출합니다. `for _ in [1, 2]`의 밑줄은 실행 횟수 값 자체를 이후 계산에 사용하지 않는다는 뜻입니다. 각 조건에서 `입력 토큰 수 + 최대 생성 20토큰`이 한도 이하인지 검사하고, 반환 문자열에 `314159`가 있으면 적중으로 셉니다. `hits=0/2`는 두 실행 모두 코드 문자열을 출력하지 않았다는 뜻이며 모델 전체의 실패율을 뜻하지 않습니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
observations = []
for repeats, position, prompt in conditions:
    outputs = []
    for _ in [1, 2]:
        output = generate(prompt, max_new_tokens=max_new_tokens)
        outputs.append(output)
    hits = 0
    for output in outputs:
        if "314159" in output:
            hits += 1
    messages = [{"role": "user", "content": prompt}]
    templated = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = len(tokenizer.encode(templated))
    fits = tokens + max_new_tokens <= model_limit
    observations.append({"repeats": repeats, "position": position, "tokens": tokens, "outputs": outputs})
    print(f"filler={repeats:2} position={position:6} tokens={tokens:4} + output={max_new_tokens} <= {model_limit}: {fits} hits={hits}/2 outputs={outputs}")

filler= 2 position=start  tokens=  99 + output=20 <= 32768: True hits=2/2 outputs=['314159', '314159']


filler= 2 position=middle tokens=  99 + output=20 <= 32768: True hits=0/2 outputs=['결제 상태: 완료\n배송 상태: 완료', '결제 상태: 완료\n배송 상태: 완료']


filler= 2 position=end    tokens=  99 + output=20 <= 32768: True hits=0/2 outputs=['결제 상태: 완료\n배송 상태: 완료', '결제 상태: 완료\n배송 상태: 완료']


filler=20 position=start  tokens= 441 + output=20 <= 32768: True hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


filler=20 position=middle tokens= 441 + output=20 <= 32768: True hits=0/2 outputs=['주문 조회 화면에는 결제 상태와 배송 상태가 표시되지 않습니다.', '주문 조회 화면에는 결제 상태와 배송 상태가 표시되지 않습니다.']


filler=20 position=end    tokens= 441 + output=20 <= 32768: True hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


filler=60 position=start  tokens=1201 + output=20 <= 32768: True hits=2/2 outputs=['314159', '314159']


filler=60 position=middle tokens=1201 + output=20 <= 32768: True hits=0/2 outputs=['주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다.', '주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다.']


filler=60 position=end    tokens=1201 + output=20 <= 32768: True hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [5]:
assert len(observations) == 9
assert model_limit == 32_768
longest_input = 0
all_have_two_outputs = True
for item in observations:
    longest_input = max(longest_input, item["tokens"])
    if len(item["outputs"]) != 2:
        all_have_two_outputs = False
assert longest_input + max_new_tokens <= model_limit
assert all_have_two_outputs
print("검증: 3개 길이 × 3개 위치가 모델 한도 안이며 중복 출력이 있음을 확인했습니다.")

검증: 3개 길이 × 3개 위치가 모델 한도 안이며 중복 출력이 있음을 확인했습니다.
